# 03 · DeepLabV3+ ResNet-101: hyperparameter tuning and Japan-only training

Source model for the zero-shot setting, trained on Japanese data only (BeachLitter v2022 + BePLi v2). Local Cirebon and Bali images are not used here.

**Labels:** 0 background, 1 artificial, 2 natural, 255 ignored (padding and out-of-image area after augmentation).

**Two phases**
1. `RUN_PHASE = "tune"`: Optuna (TPE + MedianPruner) searches the SGD learning rate and weight decay. Each trial trains for `TUNE_EPOCHS` epochs and is scored by Japanese validation mIoU. A time guard keeps each session under Kaggle's 12-hour limit; unfinished studies resume in a new session via `PREVIOUS_RUN_DIR`.
2. `RUN_PHASE = "final"`: retrains with the best parameters for `FINAL_EPOCHS` epochs, keeps the checkpoint with the best validation mIoU, and evaluates once on the Japanese test set.

**Settings**
- DeepLabV3+ (Chen et al., 2018): SGD, momentum 0.9, poly schedule (power 0.9), output stride 16.
- Search space (log scale): learning rate 1e-3 to 1e-2, weight decay 1e-5 to 1e-4.
- Optuna (Akiba et al., 2019): TPE sampler with MedianPruner.
- 512 × 512 random crop with flip, affine, colour and noise augmentation.

**Kaggle inputs:** the `beach-litter` and `databepliv2` datasets and the output of notebook 02 (`bepli_pseudo_masks/`, `v2022_split.csv`). GPU and Internet on.

## 1. Packages

In [ ]:
%pip -q install 'segmentation-models-pytorch==0.5.0' 'albumentations==2.0.8' 'optuna>=4,<5'

## 2. Configuration
`PSEUDO_RUN_DIR` is the output folder of notebook 02, containing `bepli_pseudo_masks/` and `v2022_split.csv`.

In [ ]:
import hashlib
import json
import math
import random
import re
import shutil
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import segmentation_models_pytorch as smp
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SESSION_START = time.time()

RUN_PHASE = "tune"
PREVIOUS_RUN_DIR = None

V2022_IMG = Path("/kaggle/input/datasets/rafiadabhi/beach-litter/image")
V2022_MASK = Path("/kaggle/input/datasets/rafiadabhi/beach-litter/mask")
BEPLI_DIR = Path("/kaggle/input/datasets/rafiadabhi/databepliv2")
BEPLI_LIST = BEPLI_DIR / "pembagian_anotasi_bepli.md"
PSEUDO_RUN_DIR = Path("/kaggle/input/notebooks/rafiadabhi/pseudo-mask2former-checkpoint")
BEPLI_MASK_DIR = PSEUDO_RUN_DIR / "bepli_pseudo_masks"
V2022_SPLIT_CSV = PSEUDO_RUN_DIR / "v2022_split.csv"

OUT = Path("/kaggle/working/deeplabv3plus_r101")
CACHE = Path("/tmp/cache_512")
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
CLASSES = ["background", "artificial", "natural"]
PALETTE = np.array([[0, 0, 0], [255, 0, 0], [0, 255, 0]], dtype=np.int32)
IGNORE = 255
SHORT_SIDE = 512
CROP = 512
BATCH_SIZE = 8
NUM_WORKERS = 4
ENCODER = "resnet101"
ENCODER_WEIGHTS = "imagenet"
MOMENTUM = 0.9
POLY_POWER = 0.9
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

N_TRIALS = 12
TUNE_EPOCHS = 6
FINAL_EPOCHS = 12
LR_RANGE = (1e-3, 1e-2)
WD_RANGE = (1e-5, 1e-4)
OBJECTIVE_SOURCES = ("v2022", "bepli")

START_TRIAL_LIMIT_H = 11.0
HARD_STOP_H = 11.5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"


def hours() -> float:
    return (time.time() - SESSION_START) / 3600


def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_all(SEED)
assert RUN_PHASE in {"tune", "final"}
if RUN_PHASE == "final":
    assert PREVIOUS_RUN_DIR is not None, "The final phase needs PREVIOUS_RUN_DIR with best_params.json"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print("Versions:", {"torch": torch.__version__, "smp": smp.__version__, "albumentations": A.__version__,
                 "optuna": optuna.__version__})

## 3. Data and splits
The v2022 split comes from `v2022_split.csv`, the same split used for the Mask2Former teacher, so the v2022 test images were never seen by the teacher. BePLi is split 70/10/20 with a fixed seed, stratified by quartile of natural-litter area, and saved to `bepli_split.csv`.

In [ ]:
def read_bepli_list(md_path: Path) -> list[str]:
    blocks = re.findall(r"```(.*?)```", md_path.read_text(encoding="utf-8"), re.S)
    return sorted({n for block in blocks for n in re.findall(r"\b\d{6}\b", block)})


def index_pngs(root: Path, preferred: str = "original_images") -> dict:
    index = {}
    for p in sorted(root.rglob("*.png")):
        if p.name not in index or preferred in p.parts:
            index[p.name] = p
    return index


v_split = pd.read_csv(V2022_SPLIT_CSV)
v2022 = pd.DataFrame({
    "id": "v2022_" + v_split["image"].str.replace(r"\.\w+$", "", regex=True),
    "source": "v2022",
    "image_path": [str(V2022_IMG / f) for f in v_split["image"]],
    "mask_path": [str(V2022_MASK / f) for f in v_split["mask"]],
    "mask_format": "rgb",
    "split": v_split["split"],
    "natural_pct": v_split["natural_pct"],
})

numbers = read_bepli_list(BEPLI_LIST)
png_index = index_pngs(BEPLI_DIR)
missing_img = [n for n in numbers if f"{n}.png" not in png_index]
missing_mask = [n for n in numbers if not (BEPLI_MASK_DIR / f"{n}.png").is_file()]
assert not missing_img, f"BePLi images not found: {missing_img[:5]}"
assert not missing_mask, f"BePLi pseudo-label masks not found: {missing_mask[:5]}"

bepli = pd.DataFrame({
    "id": [f"bepli_{n}" for n in numbers],
    "source": "bepli",
    "image_path": [str(png_index[f"{n}.png"]) for n in numbers],
    "mask_path": [str(BEPLI_MASK_DIR / f"{n}.png") for n in numbers],
    "mask_format": "index",
})
bepli["natural_pct"] = [
    float((cv2.resize(cv2.imread(p, cv2.IMREAD_GRAYSCALE), (128, 128),
                      interpolation=cv2.INTER_NEAREST) == 2).mean() * 100)
    for p in tqdm(bepli["mask_path"], desc="BePLi natural area")
]
strata = pd.qcut(bepli["natural_pct"].rank(method="first"), 4, labels=False)
b_train, b_rest = train_test_split(bepli.index, test_size=0.30, stratify=strata, random_state=SEED)
b_val, b_test = train_test_split(b_rest, test_size=2 / 3, stratify=strata[b_rest], random_state=SEED)
bepli["split"] = "train"
bepli.loc[b_val, "split"] = "val"
bepli.loc[b_test, "split"] = "test"

bepli_split_out = bepli.assign(image=lambda d: d["image_path"].map(lambda p: Path(p).name))[
    ["image", "split", "natural_pct"]]
if PREVIOUS_RUN_DIR is not None and (Path(PREVIOUS_RUN_DIR) / "bepli_split.csv").is_file():
    prev = pd.read_csv(Path(PREVIOUS_RUN_DIR) / "bepli_split.csv")
    same = prev[["image", "split"]].reset_index(drop=True).equals(
        bepli_split_out[["image", "split"]].reset_index(drop=True))
    assert same, "bepli_split.csv differs from the previous session; check the inputs."
bepli_split_out.to_csv(OUT / "bepli_split.csv", index=False)
shutil.copy2(V2022_SPLIT_CSV, OUT / "v2022_split.csv")

data = pd.concat([v2022, bepli], ignore_index=True)
assert data["id"].is_unique
data.to_csv(OUT / "manifest_japan.csv", index=False)
print(pd.crosstab(data["source"], data["split"], margins=True))

## 4. Cache
Images and masks are resized once so that the short side is 512 px, keeping the aspect ratio. v2022 RGB masks are mapped to the nearest palette colour, as for the Mask2Former teacher; BePLi masks are already class indices. The cache lives in `/tmp` and is not saved with the output.

In [ ]:
def rgb_to_index(mask_rgb: np.ndarray) -> np.ndarray:
    dist = ((mask_rgb[:, :, None, :].astype(np.int32) - PALETTE[None, None]) ** 2).sum(-1)
    return dist.argmin(-1).astype(np.uint8)


def load_label(path: str, fmt: str) -> np.ndarray:
    if fmt == "index":
        label = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        assert label is not None and label.max() <= 2, f"Invalid index mask: {path}"
        return label
    rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    return rgb_to_index(rgb)


def cache_one(row) -> tuple[str, int, int]:
    img_out = CACHE / f"{row.id}.jpg"
    mask_out = CACHE / f"{row.id}.png"
    if img_out.is_file() and mask_out.is_file():
        h, w = cv2.imread(str(mask_out), cv2.IMREAD_GRAYSCALE).shape
        return row.id, h, w
    img = cv2.imread(row.image_path)
    assert img is not None, f"Cannot read {row.image_path}"
    label = load_label(row.mask_path, row.mask_format)
    h, w = img.shape[:2]
    if label.shape != (h, w):
        label = cv2.resize(label, (w, h), interpolation=cv2.INTER_NEAREST)
    scale = SHORT_SIDE / min(h, w)
    size = (round(w * scale), round(h * scale))
    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    cv2.imwrite(str(img_out), cv2.resize(img, size, interpolation=interp), [cv2.IMWRITE_JPEG_QUALITY, 95])
    cv2.imwrite(str(mask_out), cv2.resize(label, size, interpolation=cv2.INTER_NEAREST))
    return row.id, size[1], size[0]


CACHE.mkdir(parents=True, exist_ok=True)
t0 = time.time()
with ThreadPoolExecutor(max_workers=4) as pool:
    results = list(tqdm(pool.map(cache_one, data.itertuples(index=False)), total=len(data), desc="Cache"))
print(f"Cache ready: {len(results)} pairs in {(time.time() - t0) / 60:.1f} min")

counts = np.zeros(3)
for row in data[data.split == "train"].itertuples(index=False):
    counts += np.bincount(cv2.imread(str(CACHE / f"{row.id}.png"), cv2.IMREAD_GRAYSCALE).ravel(), minlength=3)[:3]
print("Class pixel share (train):", dict(zip(CLASSES, (counts / counts.sum()).round(4))))

## 5. Dataset and augmentation

In [ ]:
def train_transform() -> A.Compose:
    return A.Compose([
        A.PadIfNeeded(min_height=CROP, min_width=CROP, border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=IGNORE),
        A.RandomCrop(height=CROP, width=CROP),
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.85, 1.15), translate_percent=(-0.05, 0.05), rotate=(-7, 7),
                 border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=IGNORE, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
        A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=20, val_shift_limit=15, p=0.3),
        A.GaussNoise(std_range=(0.02, 0.08), p=0.2),
        A.Normalize(mean=MEAN, std=STD),
    ])


def eval_transform() -> A.Compose:
    return A.Compose([A.Normalize(mean=MEAN, std=STD)])


class JapanDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, training: bool):
        self.ids = frame["id"].tolist()
        self.sources = frame["source"].tolist()
        self.transform = train_transform() if training else eval_transform()

    def __len__(self) -> int:
        return len(self.ids)

    def __getitem__(self, i):
        image = cv2.cvtColor(cv2.imread(str(CACHE / f"{self.ids[i]}.jpg")), cv2.COLOR_BGR2RGB)
        label = cv2.imread(str(CACHE / f"{self.ids[i]}.png"), cv2.IMREAD_GRAYSCALE)
        out = self.transform(image=image, mask=label)
        image = torch.from_numpy(np.ascontiguousarray(out["image"].transpose(2, 0, 1))).float()
        return image, torch.from_numpy(out["mask"].astype(np.int64)), self.sources[i]


def seed_worker(worker_id: int) -> None:
    seed = torch.initial_seed() % 2**32
    random.seed(seed)
    np.random.seed(seed)
    torch.utils.data.get_worker_info().dataset.transform.set_random_seed(seed)


def make_loader(frame: pd.DataFrame, training: bool, seed: int) -> DataLoader:
    dataset = JapanDataset(frame, training)
    dataset.transform.set_random_seed(seed)
    return DataLoader(
        dataset, batch_size=BATCH_SIZE if training else 1, shuffle=training, drop_last=training,
        num_workers=NUM_WORKERS, worker_init_fn=seed_worker, pin_memory=DEVICE.type == "cuda",
        generator=torch.Generator().manual_seed(seed), persistent_workers=False,
    )


train_df = data[data.split == "train"].reset_index(drop=True)
val_df = data[data.split == "val"].reset_index(drop=True)
test_df = data[data.split == "test"].reset_index(drop=True)
print(f"train={len(train_df)} | val={len(val_df)} | test={len(test_df)}")

## 6. Model, loss, optimizer, and metrics
Loss = cross-entropy (ignoring 255) + soft Dice on the artificial and natural classes. Evaluation runs on whole images (short side 512) padded to a multiple of 32; padding is ignored. Area MAE is the absolute difference between predicted and true class area, in percentage points of valid pixels.

In [ ]:
def build_model(weights: str | None = ENCODER_WEIGHTS) -> nn.Module:
    return smp.DeepLabV3Plus(encoder_name=ENCODER, encoder_weights=weights,
                             encoder_output_stride=16, classes=3, activation=None).to(DEVICE)


ce_loss = nn.CrossEntropyLoss(ignore_index=IGNORE)


def criterion(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    ce = ce_loss(logits, targets)
    prob = torch.softmax(logits.float(), dim=1)
    valid = (targets != IGNORE).float()
    terms = []
    for c in (1, 2):
        actual = (targets == c).float()
        if actual.sum() == 0:
            continue
        pred = prob[:, c] * valid
        terms.append(1 - (2 * (pred * actual).sum() + 1) / (pred.sum() + actual.sum() + 1))
    return ce + (torch.stack(terms).mean() if terms else ce.new_zeros(()))


def make_optimizer(model: nn.Module, lr: float, wd: float, total_steps: int):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda s: (1 - min(s, total_steps) / total_steps) ** POLY_POWER)
    return optimizer, scheduler


def pad32(image: torch.Tensor, label: torch.Tensor):
    h, w = image.shape[-2:]
    ph, pw = (-h) % 32, (-w) % 32
    if ph or pw:
        image = F.pad(image, (0, pw, 0, ph), value=0.0)
        label = F.pad(label, (0, pw, 0, ph), value=IGNORE)
    return image, label


@torch.no_grad()
def evaluate(model: nn.Module, frame: pd.DataFrame, timing: bool = False) -> dict:
    model.eval()
    conf = {}
    area_err = {}
    times = []
    for image, label, source in make_loader(frame, training=False, seed=SEED):
        source = source[0]
        image, label = pad32(image.to(DEVICE), label.to(DEVICE))
        if timing and DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.autocast(device_type="cuda", enabled=USE_AMP):
            pred = model(image).argmax(1)
        if timing and DEVICE.type == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
        valid = label != IGNORE
        t, p = label[valid], pred[valid]
        cm = torch.bincount(t * 3 + p, minlength=9).reshape(3, 3).cpu().numpy()
        conf[source] = conf.get(source, 0) + cm
        n = valid.sum().item()
        area_err.setdefault(source, []).append(
            [abs((p == c).sum().item() - (t == c).sum().item()) / n * 100 for c in (1, 2)])
    return {"conf": conf, "area_err": {s: np.mean(v, axis=0) for s, v in area_err.items()},
            "ms_per_image": float(np.median(times)) if timing else None}


def iou_from_conf(cm: np.ndarray) -> np.ndarray:
    tp = np.diag(cm).astype(float)
    return tp / np.maximum(cm.sum(0) + cm.sum(1) - tp, 1)


def dice_from_conf(cm: np.ndarray) -> np.ndarray:
    tp = np.diag(cm).astype(float)
    return 2 * tp / np.maximum(cm.sum(0) + cm.sum(1), 1)


def miou(result: dict, sources) -> float:
    cm = sum(result["conf"][s] for s in sources if s in result["conf"])
    return float(iou_from_conf(cm).mean())


def summarize(result: dict) -> pd.DataFrame:
    rows = []
    groups = {s: [s] for s in sorted(result["conf"])}
    groups["japan"] = sorted(result["conf"])
    for name, sources in groups.items():
        cm = sum(result["conf"][s] for s in sources)
        iou, dice = iou_from_conf(cm), dice_from_conf(cm)
        row = {"group": name, "mIoU": iou.mean(), "mDice": dice.mean()}
        row.update({f"IoU_{c}": v for c, v in zip(CLASSES, iou)})
        row.update({f"Dice_{c}": v for c, v in zip(CLASSES, dice)})
        if name in result["area_err"]:
            row.update({"area_MAE_artificial_pp": result["area_err"][name][0],
                        "area_MAE_natural_pp": result["area_err"][name][1]})
        rows.append(row)
    return pd.DataFrame(rows).round(4)

## 7. Training
Shared by tuning and final training. With an Optuna `trial`, validation mIoU is reported each epoch for pruning.

In [ ]:
def train_model(lr: float, wd: float, epochs: int, seed: int, trial=None, ckpt_path: Path | None = None) -> dict:
    seed_all(seed)
    model = build_model()
    loader = make_loader(train_df, training=True, seed=seed)
    optimizer, scheduler = make_optimizer(model, lr, wd, epochs * len(loader))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best = {"miou": -1.0, "epoch": 0}
    history = []
    for epoch in range(1, epochs + 1):
        if hours() > HARD_STOP_H:
            print(f"Hard stop at {HARD_STOP_H} h; stopped before epoch {epoch}.")
            best["stopped_by_time"] = True
            break
        model.train()
        t0, loss_sum, n_batches = time.time(), 0.0, 0
        for images, targets, _ in tqdm(loader, desc=f"epoch {epoch}/{epochs}", leave=False):
            images, targets = images.to(DEVICE, non_blocking=True), targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=USE_AMP):
                loss = criterion(model(images), targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            loss_sum += loss.item()
            n_batches += 1
        result = evaluate(model, val_df)
        score = miou(result, OBJECTIVE_SOURCES)
        score_v2022 = miou(result, ["v2022"])
        history.append({"epoch": epoch, "loss": loss_sum / max(n_batches, 1), "val_miou": score,
                        "val_miou_v2022": score_v2022, "seconds": time.time() - t0})
        print(f"lr={lr:.2e} wd={wd:.2e} | epoch {epoch}/{epochs} | loss {history[-1]['loss']:.4f} | "
              f"val mIoU {score:.4f} (v2022 {score_v2022:.4f}) | {history[-1]['seconds'] / 60:.1f} min | "
              f"session {hours():.2f} h", flush=True)
        if score > best["miou"]:
            best.update(miou=score, epoch=epoch, val_summary=summarize(result).to_dict("records"))
            if ckpt_path is not None:
                torch.save(model.state_dict(), ckpt_path)
        if trial is not None:
            trial.report(score, epoch)
            if trial.should_prune():
                best["history"] = history
                trial.set_user_attr("history", history)
                raise optuna.TrialPruned()
    best["history"] = history
    del model, optimizer
    torch.cuda.empty_cache()
    return best

## 8. Tuning (`RUN_PHASE = "tune"`)
Before each trial, its duration is estimated from earlier trials, including those from previous sessions. A trial starts only if it is expected to finish before `START_TRIAL_LIMIT_H`. Results are saved after every trial. The first five trials are random, the rest guided by TPE; the sampler seed is offset by the number of existing trials so a resumed session does not repeat the same random points.

In [ ]:
config_signature = hashlib.sha256(json.dumps({
    "encoder": ENCODER, "short_side": SHORT_SIDE, "crop": CROP, "batch": BATCH_SIZE,
    "tune_epochs": TUNE_EPOCHS, "lr_range": LR_RANGE, "wd_range": WD_RANGE, "objective": OBJECTIVE_SOURCES,
    "n_train": len(train_df), "n_val": len(val_df), "seed": SEED, "aug": "tabel_2_2_v1",
}, sort_keys=True).encode()).hexdigest()[:10]
STUDY_NAME = f"deeplabv3plus_r101_{config_signature}"
STORAGE = f"sqlite:///{OUT / 'study.db'}"


def save_study_outputs(study: optuna.Study) -> None:
    study.trials_dataframe().to_csv(OUT / "optuna_trials.csv", index=False)
    done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if done:
        best = study.best_trial
        payload = {"study": STUDY_NAME, "signature": config_signature, "value": best.value,
                   "params": best.params, "best_epoch": best.user_attrs.get("best_epoch"),
                   "finished_trials": sum(t.state.is_finished() for t in study.trials)}
        (OUT / "best_params.json").write_text(json.dumps(payload, indent=2))


def objective(trial: optuna.Trial) -> float:
    lr = trial.suggest_float("learning_rate", *LR_RANGE, log=True)
    wd = trial.suggest_float("weight_decay", *WD_RANGE, log=True)
    t0 = time.time()
    try:
        best = train_model(lr, wd, TUNE_EPOCHS, seed=SEED, trial=trial)
    finally:
        trial.set_user_attr("duration_h", (time.time() - t0) / 3600)
    trial.set_user_attr("best_epoch", best["epoch"])
    trial.set_user_attr("history", best["history"])
    if best.get("stopped_by_time"):
        raise optuna.TrialPruned()
    return best["miou"]


if RUN_PHASE == "tune":
    if PREVIOUS_RUN_DIR is not None:
        src = Path(PREVIOUS_RUN_DIR) / "study.db"
        assert src.is_file(), f"No study.db in {PREVIOUS_RUN_DIR}"
        shutil.copy2(src, OUT / "study.db")
    study = optuna.create_study(
        study_name=STUDY_NAME, storage=STORAGE, load_if_exists=True, direction="maximize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=2),
    )
    if PREVIOUS_RUN_DIR is not None and not study.trials:
        raise ValueError("study.db has no study with this configuration; keep the configuration fixed across sessions.")
    study.sampler = optuna.samplers.TPESampler(seed=SEED + len(study.trials), n_startup_trials=5)

    while True:
        finished = [t for t in study.trials if t.state.is_finished()]
        if len(finished) >= N_TRIALS:
            print(f"Reached {N_TRIALS} trials.")
            break
        durations = [t.user_attrs["duration_h"] for t in finished if "duration_h" in t.user_attrs]
        estimate = max(durations) * 1.1 if durations else None
        if estimate is not None and hours() + estimate > START_TRIAL_LIMIT_H:
            print(f"Not enough time for another trial (estimate {estimate:.2f} h). "
                  f"Continue in a new session with PREVIOUS_RUN_DIR set to this output folder.")
            break
        print(f"\n=== Trial {len(finished) + 1}/{N_TRIALS} | session {hours():.2f} h ===")
        study.optimize(objective, n_trials=1, gc_after_trial=True)
        save_study_outputs(study)

    save_study_outputs(study)
    finished = sum(t.state.is_finished() for t in study.trials)
    print(f"Finished trials: {finished}/{N_TRIALS}")
    if any(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials):
        print("Best:", study.best_trial.params, "| val mIoU:", round(study.best_value, 4))
    display(study.trials_dataframe()[["number", "state", "value", "params_learning_rate",
                                      "params_weight_decay", "user_attrs_duration_h"]])
else:
    print("RUN_PHASE = final: tuning skipped.")

## 9. Final training and test evaluation (`RUN_PHASE = "final"`)
Uses `best_params.json` from the tuning session. The checkpoint with the best validation mIoU is evaluated once on the Japanese test set. BePLi test scores are reported separately: its natural labels are pseudo-labels, so they measure agreement with the pseudo-labels, not accuracy against ground truth.

In [ ]:
if RUN_PHASE == "final":
    payload = json.loads((Path(PREVIOUS_RUN_DIR) / "best_params.json").read_text())
    for name in ("best_params.json", "optuna_trials.csv"):
        if (Path(PREVIOUS_RUN_DIR) / name).is_file():
            shutil.copy2(Path(PREVIOUS_RUN_DIR) / name, OUT / name)
    assert payload["signature"] == config_signature, "Configuration differs from the tuning session."
    lr, wd = payload["params"]["learning_rate"], payload["params"]["weight_decay"]
    print(f"Best parameters: lr={lr:.3e}, wd={wd:.3e} (tuning mIoU {payload['value']:.4f})")

    CKPT = OUT / "deeplabv3plus_r101_best.pt"
    final = train_model(lr, wd, FINAL_EPOCHS, seed=SEED, ckpt_path=CKPT)
    pd.DataFrame(final["history"]).to_csv(OUT / "history_final.csv", index=False)
    assert CKPT.is_file(), "Checkpoint was not saved."

    model = build_model(weights=None)
    model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
    test_result = evaluate(model, test_df, timing=True)
    test_table = summarize(test_result)
    test_table.to_csv(OUT / "test_metrics.csv", index=False)

    metrics = {
        "model": "DeepLabV3+", "encoder": ENCODER, "lr": lr, "weight_decay": wd,
        "best_epoch": final["epoch"], "val_miou_objective": final["miou"],
        "test": test_table.to_dict("records"), "ms_per_image_median": test_result["ms_per_image"],
        "note": "BePLi test scores measure agreement with pseudo-labels (natural class).",
    }
    (OUT / "metrics_final.json").write_text(json.dumps(metrics, indent=2))
    print(f"Best epoch: {final['epoch']} | median inference time: {test_result['ms_per_image']:.1f} ms/image")
    display(test_table)
else:
    print("RUN_PHASE = tune: final training skipped.")

## 10. Output
Save the output folder or attach it as input to the next session. `v2022_split.csv` and `bepli_split.csv` are shared with notebook 04.

In [ ]:
print(f"Session time: {hours():.2f} h")
for f in sorted(OUT.iterdir()):
    print(f"{f.name:35s} {f.stat().st_size / 1e6:8.2f} MB")